[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%205/L9_Demo_SteamReviewAssistant.ipynb)

# ISBA 2411 · Week 5 · Lecture 9
## Should You Play It? — a Steam Game-Review Assistant

Last week you *built* a neural network from scratch. Tonight you *borrow* pretrained ones off **Hugging Face** and point them at thousands of real **Steam game reviews** — no training, three lines each.

Tonight's tools:
- **Ask the reviews** — *question answering*: “how hard is Sekiro?” → it extracts the answer from real reviews.
- **Review Router** — *zero-shot classification*: sort any review into categories **you invent on the spot**.
- **Multi-lens work-up** — *sentiment · emotion · irony · toxicity* on any pasted review, all at once.

**The one idea that ties it together — the `pipeline`.** Every model below is loaded the same way: `pipeline(task)` wraps tokenizing your text, running a pretrained model, and decoding the result into an answer. You write one line; it does the rest.

<svg viewBox="0 0 690 170" width="100%" style="max-width:690px;font-family:system-ui">
<rect width="690" height="170" rx="12" fill="#0f151d"/>
<text x="16" y="26" fill="#8ea0b3" font-size="12" font-family="monospace" letter-spacing="1.5">HOW A pipeline() WORKS</text>
<g font-size="13" text-anchor="middle">
<rect x="14" y="46" width="112" height="52" rx="8" fill="#1b2430" stroke="#2d3a4c"/><text x="70" y="70" fill="#dbe4ee">raw text</text><text x="70" y="87" fill="#61738a" font-size="11">a review</text>
<rect x="160" y="46" width="112" height="52" rx="8" fill="#1b2430" stroke="#2d3a4c"/><text x="216" y="70" fill="#dbe4ee">tokenize</text><text x="216" y="87" fill="#61738a" font-size="11">text &#8594; ids</text>
<rect x="306" y="46" width="112" height="52" rx="8" fill="#1b2430" stroke="#57b0e6"/><text x="362" y="70" fill="#dbe4ee">model</text><text x="362" y="87" fill="#61738a" font-size="11">pretrained</text>
<rect x="452" y="46" width="112" height="52" rx="8" fill="#1b2430" stroke="#2d3a4c"/><text x="508" y="70" fill="#dbe4ee">decode</text><text x="508" y="87" fill="#61738a" font-size="11">ids &#8594; label</text>
<rect x="598" y="46" width="78" height="52" rx="8" fill="#152130" stroke="#a6cf46"/><text x="637" y="76" fill="#a6cf46">output</text></g>
<g fill="#e0a24e" font-size="18"><text x="140" y="76">&#8594;</text><text x="286" y="76">&#8594;</text><text x="432" y="76">&#8594;</text><text x="580" y="76">&#8594;</text></g>
<path d="M160 118 L160 128 L564 128 L564 118" fill="none" stroke="#57b0e6" stroke-width="1.2"/>
<text x="362" y="150" fill="#57b0e6" font-size="12.5" text-anchor="middle" font-family="monospace">pipeline(task) does all three &#8212; you call one function</text></svg>

> 🚀 **This whole notebook is a template for your final project.** Load a dataset → borrow pretrained models → retrieve-then-read → wrap it in a demo UI. Swap the Steam reviews for your project's data and the skeleton is the same.

---
**Runtime:** Colab. Free CPU works; **GPU is much faster** for the Router (Runtime → Change runtime type → GPU).

## 0 · Setup

**What this cell does.** Installs the libraries and imports `pipeline` — the single function we use to load every model tonight. The version floors are deliberately loose so Colab never has to downgrade a package (a downgrade would need a runtime restart).

> 🎓 **Class connection.** `transformers`, `gradio`, and `scikit-learn` are the same toolkit from Weeks 3–4 — you're now composing them, not meeting them.

In [ ]:
%pip install -q "transformers>=4.40" "gradio>=4.44" scikit-learn pandas matplotlib "qrcode[pil]"

import pandas as pd, re, html, textwrap
from transformers import pipeline
print('setup complete')

## 1 · Load the Steam reviews

**What this cell does.** Reads ~5,000 English reviews across 16 games into a `DataFrame`, tidies the text (collapses whitespace, unescapes HTML), and keeps each game's Steam **app-id** so we can show cover art later. Crucially, each row carries the player's real **thumbs-up/down** (`recommended`) — that's *ground-truth labels we didn't have to make.*

> 🎓 **Class connection.** This is the *corpus + data-cleaning* step from Weeks 2–3 — the unglamorous 80% of real NLP.

> 🚀 **Final project.** Your first milestone task: get your data into a tidy `DataFrame` with a text column and, if you're lucky, a label column. Everything downstream depends on this being clean.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/steam_reviews_demo.csv'

df = pd.read_csv(DATA_URL)
df['review'] = df['review'].map(lambda t: re.sub(r'\s+', ' ', html.unescape(str(t))).strip())
df = df[df['review'].str.len() > 40].reset_index(drop=True)

APPIDS = {'Elden Ring':1245620, "Baldur's Gate 3":1086940, 'Cyberpunk 2077':1091500,
          'Stardew Valley':413150, 'The Witcher 3':292030, 'Counter-Strike 2':730,
          'Terraria':105600, 'Hollow Knight':367520, 'Hades':1145360, 'Portal 2':620,
          'Grand Theft Auto V':271590, 'Red Dead Redemption 2':1174180, 'Sekiro':814380,
          "No Man's Sky":275850, 'DOOM Eternal':782330, 'Team Fortress 2':440}
def cover(game):
    aid = APPIDS.get(game)
    return f'https://cdn.cloudflare.steamstatic.com/steam/apps/{aid}/header.jpg' if aid else None

print(f'{len(df):,} reviews | {df.game.nunique()} games')
df.sample(3)

## 2 · Warm-up — sentiment, and a reality check

**What this cell does.** Runs a sentiment `pipeline` on 40 reviews and — because we have the player's thumb as ground truth — measures how often the model **agrees with reality**, then prints the misses and plots a confusion matrix.

**Why the confusion matrix matters.** Accuracy alone hides *where* a model fails. The matrix splits it into the four cases (right-positive, right-negative, and the two kinds of mistake) so you can see the model's bias at a glance.

> 🎓 **Class connection.** Sentiment was introduced in Week 4; the confusion matrix is the Week-3 evaluation habit — never trust a model without checking it against ground truth.

> 🚀 **Final project.** This is exactly how you'll **evaluate** your system: hold out labeled examples, compare predictions to truth, and *show the confusion matrix* — graders reward honest error analysis over a single accuracy number.

In [ ]:
sentiment = pipeline('sentiment-analysis')

sample = df.sample(40, random_state=0).copy()
sample['model'] = [sentiment(r[:512])[0]['label'] for r in sample['review']]
sample['player'] = sample['recommended'].map({'Recommended':'POSITIVE','Not Recommended':'NEGATIVE'})
agree = (sample['model'] == sample['player']).mean()
print(f'Model agrees with the player thumb {agree:.0%} of the time (n=40)')

print('\n--- a few misses (model vs player) ---')
miss = sample[sample['model'] != sample['player']].head(4)
for r in miss.itertuples():
    print(f"model={r.model:8} player={r.player:8} | {textwrap.shorten(r.review, 90)}")

# --- confusion matrix: where sentiment agrees/disagrees with the player thumb ---
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(sample['player'], sample['model'], labels=['POSITIVE','NEGATIVE'])
fig, ax = plt.subplots(figsize=(3.4, 3))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_xticklabels(['pos','neg'])
ax.set_yticks([0,1]); ax.set_yticklabels(['pos','neg'])
ax.set_xlabel('model says'); ax.set_ylabel('player thumb (truth)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=13)
ax.set_title('sentiment vs. ground truth'); plt.tight_layout(); plt.show()

## 3 · Two models, one review — which do you trust?

**What this cell does.** Runs the same tricky reviews through two *different* pretrained sentiment models — one trained on movie reviews, one on product reviews — and lines up their answers. Where they disagree is where model choice actually matters.

> 🎓 **Class connection.** This is the Week-3 *model bake-off* idea: the best model is domain-dependent, and the only way to know is to compare on *your* data.

> 🚀 **Final project.** Before you commit to a model, run two or three candidates on a sample and compare — a five-minute bake-off saves you from shipping the wrong one.

In [ ]:
stars = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')
def star_label(t):
    n = int(stars(t[:512])[0]['label'][0])   # '1 star'..'5 stars'
    return 'POSITIVE' if n >= 4 else 'NEGATIVE' if n <= 2 else 'NEUTRAL'

tricky = ['10/10 would rage quit again.',
          "It's not a bad game.",
          'Great game if you enjoy losing your entire weekend.',
          'I have 400 hours and I hate every second.',
          'Peak. Simply peak.']
print(f"{'review':45} {'SST-2':10} {'review-model'}")
for t in tricky:
    print(f"{textwrap.shorten(t,43):45} {sentiment(t)[0]['label']:10} {star_label(t)}")

## 4 · Tool A — Review Router (zero-shot classification)

**What this cell does.** Loads one pretrained model and sorts a review into *any* categories you type at call time — `graphics`, `difficulty`, `price` — printing a score for each. **Nothing is trained.**

**The algorithm (zero-shot via NLI).** The model was pretrained on *natural-language inference* (does sentence A imply sentence B?). Zero-shot reuses that: for each label it builds the hypothesis *“this review is about &lt;label&gt;”* and scores how strongly the review **entails** it.

<svg viewBox="0 0 690 150" width="100%" style="max-width:690px;font-family:system-ui">
<rect width="690" height="150" rx="12" fill="#0f151d"/>
<text x="16" y="24" fill="#8ea0b3" font-size="12" font-family="monospace" letter-spacing="1.5">ZERO-SHOT = ENTAILMENT (no training)</text>
<g font-size="12.5" text-anchor="middle">
<rect x="12" y="44" width="120" height="54" rx="8" fill="#1b2430" stroke="#2d3a4c"/><text x="72" y="66" fill="#dbe4ee" font-size="11.5">the review</text><text x="72" y="84" fill="#61738a" font-size="10.5">"ran like garbage"</text>
<rect x="150" y="52" width="104" height="36" rx="8" fill="#152130" stroke="#c98bdc"/><text x="202" y="75" fill="#e2c9f0">your label</text>
<rect x="300" y="44" width="150" height="54" rx="8" fill="#1b2430" stroke="#2d3a4c"/><text x="375" y="65" fill="#dbe4ee" font-size="11">hypothesis:</text><text x="375" y="82" fill="#8ea0b3" font-size="10">"this is about ___"</text>
<rect x="470" y="44" width="96" height="54" rx="8" fill="#1b2430" stroke="#57b0e6"/><text x="518" y="68" fill="#dbe4ee" font-size="11.5">NLI model</text><text x="518" y="85" fill="#61738a" font-size="10">entails?</text>
<rect x="584" y="52" width="94" height="36" rx="8" fill="#152130" stroke="#a6cf46"/><text x="631" y="75" fill="#a6cf46" font-family="monospace">0.71</text></g>
<g fill="#e0a24e" font-size="16"><text x="270" y="76">&#8594;</text><text x="456" y="76">&#8594;</text><text x="572" y="76">&#8594;</text></g>
<text x="345" y="128" fill="#61738a" font-size="11.5" text-anchor="middle">repeat for every label you invent &#8212; the entailment probability is the score</text></svg>

> 🎓 **Class connection.** Week 3 you built a *supervised* classifier — classes fixed up front, trained on labels. This is the opposite: **no training, no fixed label set**, same `pipeline` pattern.

> 🚀 **Final project.** Zero-shot is the fastest way to **classify before you have labeled data** — tag your documents on day one, eyeball the results, and only invest in labeling/fine-tuning if you need more accuracy.

In [ ]:
router = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

review = 'Gorgeous game but it ran like garbage on my PC for the first month. Fixed now, worth it.'
labels = ['graphics', 'performance', 'price or value', 'difficulty', 'story']
r = router(review, candidate_labels=labels, multi_label=True)
for lab, sc in zip(r['labels'], r['scores']):
    print(f"{lab:16} {'#'*int(sc*20):20} {sc:.2f}")

### The same review, a different question

**What this cell does.** Runs the *same* review through two different label sets. Same text, different answer — because **the labels are the question you ask the data.** Choosing them well is the real skill.

In [ ]:
for lab in [['a complaint','a recommendation','a bug report'],
            ['about the price','about the graphics','about the story']]:
    r = router(review, candidate_labels=lab, multi_label=False)
    print(lab, '->', r['labels'][0], f"({r['scores'][0]:.2f})")

## 5 · Tool B — Ask the reviews (extractive QA)

**What this cell does.** Loads a question-answering model and, given a **question** + some **review text** as context, returns the exact **span** of the text that answers it — plus a confidence score. *(It auto-falls-back to a direct model load so it works on any recent `transformers` version.)*

**The algorithm (span extraction).** The model scores every token twice — how likely it is the **start** of the answer, and the **end**. The answer is the span between the best start and best end. It can only return words that are already there, so it never hallucinates — but it also can't answer what the text doesn't say.

<svg viewBox="0 0 690 175" width="100%" style="max-width:690px;font-family:system-ui">
<rect width="690" height="175" rx="12" fill="#0f151d"/>
<text x="16" y="24" fill="#8ea0b3" font-size="12" font-family="monospace" letter-spacing="1.5">EXTRACTIVE QA = PICK A SPAN</text>
<text x="16" y="46" fill="#61738a" font-size="11.5">question: &#8220;how hard is it?&#8221;  +  the retrieved reviews as context  &#8594;</text>
<g font-size="12" text-anchor="middle" fill="#dbe4ee">
<rect x="60" y="58" width="72" height="30" rx="6" fill="#1b2430" stroke="#2d3a4c"/><text x="96" y="78">Elden</text>
<rect x="140" y="58" width="58" height="30" rx="6" fill="#1b2430" stroke="#2d3a4c"/><text x="169" y="78">Ring</text>
<rect x="206" y="58" width="40" height="30" rx="6" fill="#1b2430" stroke="#2d3a4c"/><text x="226" y="78">is</text>
<rect x="254" y="58" width="82" height="30" rx="6" fill="#152130" stroke="#57b0e6"/><text x="295" y="78">brutal</text>
<rect x="344" y="58" width="52" height="30" rx="6" fill="#152130" stroke="#57b0e6"/><text x="370" y="78">but</text>
<rect x="404" y="58" width="60" height="30" rx="6" fill="#152130" stroke="#57b0e6"/><text x="434" y="78">fair</text>
<rect x="472" y="58" width="34" height="30" rx="6" fill="#1b2430" stroke="#2d3a4c"/><text x="489" y="78">.</text></g>
<text x="295" y="106" fill="#a6cf46" font-size="11" text-anchor="middle">&#9650; start</text>
<text x="434" y="106" fill="#e0a24e" font-size="11" text-anchor="middle">&#9650; end</text>
<path d="M254 118 L254 126 L464 126 L464 118" fill="none" stroke="#57b0e6" stroke-width="1.2"/>
<text x="359" y="144" fill="#57b0e6" font-size="12" text-anchor="middle">answer span: &#8220;brutal but fair&#8221;</text>
<text x="345" y="165" fill="#61738a" font-size="11" text-anchor="middle">the model picks a start &amp; end token &#8212; it can only return words already there (no hallucination)</text></svg>

> 🎓 **Class connection.** This is *reading comprehension* — a different head on the same transformer family you'll formalize in Week 7.

> 🚀 **Final project.** Extractive QA is the **“read”** half of a retrieval-augmented system (RAG). Paired with the retrieval below, it's a hallucination-resistant way to answer questions over *your* documents — the core of a strong Week-8/final build.

In [ ]:
# QA that works on transformers 4.x AND 5.x (v5 removed the 'question-answering' pipeline string)
_QA_NAME = 'distilbert-base-cased-distilled-squad'
try:
    qa = pipeline('question-answering', model=_QA_NAME)          # transformers 4.x
except (KeyError, ValueError):
    import torch
    from transformers import AutoTokenizer, AutoModelForQuestionAnswering
    _qtok = AutoTokenizer.from_pretrained(_QA_NAME)              # transformers 5.x fallback
    _qmdl = AutoModelForQuestionAnswering.from_pretrained(_QA_NAME)
    def qa(question, context):
        inp = _qtok(question, context, return_tensors='pt', truncation=True, max_length=384)
        with torch.no_grad(): out = _qmdl(**inp)
        s = int(out.start_logits.argmax()); e = int(out.end_logits.argmax()) + 1
        ans = _qtok.decode(inp['input_ids'][0][s:e], skip_special_tokens=True)
        score = float(torch.softmax(out.start_logits, -1).max() * torch.softmax(out.end_logits, -1).max())
        return {'answer': ans, 'score': score}

context = ('Elden Ring is brutal but fair. The open world is stunning, though the '
           'difficulty spikes and some late bosses feel unfair.')
for q in ['How hard is the game?', 'What do people like about it?']:
    a = qa(question=q, context=context)
    print(f"Q: {q}\n   A: {a['answer']!r}  (score {a['score']:.2f})\n")

### Give QA the *right* reviews to read (Week-4 callback)

**What this cell does.** Builds a **TF-IDF** index of every review and, for a question, ranks reviews by **cosine similarity** to pull the most relevant few — filtered to a specific game if the question names one. Those become the context we hand to QA.

**The algorithm (retrieve then read).** Keyword search finds *what to read*; the QA model does *the reading*. Feeding the model only the relevant reviews is what makes the answer good.

<svg viewBox="0 0 690 180" width="100%" style="max-width:690px;font-family:system-ui">
<rect width="690" height="180" rx="12" fill="#0f151d"/>
<text x="16" y="24" fill="#8ea0b3" font-size="12" font-family="monospace" letter-spacing="1.5">RETRIEVAL: TF-IDF + COSINE (WEEK 4)</text>
<g font-size="12" text-anchor="middle" fill="#dbe4ee">
<rect x="14" y="44" width="140" height="38" rx="7" fill="#1b2430" stroke="#2d3a4c"/><text x="84" y="67">"how hard is it?"</text>
<rect x="14" y="104" width="140" height="38" rx="7" fill="#1b2430" stroke="#2d3a4c"/><text x="84" y="127">5,087 reviews</text>
<rect x="196" y="44" width="110" height="38" rx="7" fill="#1b2430" stroke="#2d3a4c"/><text x="251" y="67">query vector</text>
<rect x="196" y="104" width="110" height="38" rx="7" fill="#1b2430" stroke="#2d3a4c"/><text x="251" y="127">TF-IDF matrix</text>
<rect x="352" y="74" width="112" height="38" rx="7" fill="#1b2430" stroke="#57b0e6"/><text x="408" y="97">cosine sim</text>
<rect x="500" y="74" width="130" height="38" rx="7" fill="#152130" stroke="#a6cf46"/><text x="565" y="97">top-3 reviews</text></g>
<g fill="#e0a24e" font-size="15"><text x="172" y="68">&#8594;</text><text x="172" y="128">&#8594;</text><text x="330" y="97">&#8594;</text><text x="478" y="97">&#8594;</text></g>
<text x="345" y="164" fill="#61738a" font-size="11.5" text-anchor="middle">keyword search picks <tspan fill="#a6cf46">what to read</tspan>; the QA model does the reading</text></svg>

> 🎓 **Class connection.** TF-IDF and cosine similarity are straight from the Week-4 retrieval / semantic-search lecture — here they're a component, not the whole show.

> 🚀 **Final project.** **Retrieve → read** is the RAG backbone. Swap TF-IDF for embeddings (Week 6) and this same two-step is a production question-answering pipeline over your corpus.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

vec = TfidfVectorizer(stop_words='english', max_features=6000)
M = vec.fit_transform(df['review'])
GAMES = sorted(df['game'].unique())
ALIASES = {'bg3':"Baldur's Gate 3",'baldur':"Baldur's Gate 3",'gta':'Grand Theft Auto V',
           'rdr2':'Red Dead Redemption 2','red dead':'Red Dead Redemption 2','cs2':'Counter-Strike 2',
           'counter strike':'Counter-Strike 2','tf2':'Team Fortress 2','witcher':'The Witcher 3',
           'doom':'DOOM Eternal','stardew':'Stardew Valley','no mans sky':"No Man's Sky"}

def detect_game(text):
    t = text.lower()
    for g in GAMES:
        if g.lower() in t: return g
    for k, v in ALIASES.items():
        if k in t: return v
    return None

def top_reviews(query, k=3, game=None):
    idx = df.index[df['game'] == game] if game else df.index
    sims = linear_kernel(vec.transform([query]), M[idx]).ravel()
    return df.loc[idx[sims.argsort()[::-1][:k]], 'review'].tolist()

print(top_reviews('how hard is it', k=1, game='Sekiro')[0][:200])

## 6 · Three more lenses — emotion · irony · toxicity

**What this cell does.** Loads three more text-classification `pipeline`s so any review can get a full **work-up**: what emotion it carries, whether it's ironic, and whether it's toxic. Same three-line pattern, three different pretrained models — each with its own blind spots.

**The idea (an ensemble of specialists).** No single model sees everything. Running several *specialist* models over the same text gives you complementary views — and when they disagree, that tension is informative.

<svg viewBox="0 0 690 190" width="100%" style="max-width:690px;font-family:system-ui">
<rect width="690" height="190" rx="12" fill="#0f151d"/>
<text x="16" y="24" fill="#8ea0b3" font-size="12" font-family="monospace" letter-spacing="1.5">MULTI-LENS: FOUR MODELS, ONE REVIEW</text>
<rect x="14" y="72" width="120" height="46" rx="8" fill="#1b2430" stroke="#2d3a4c"/><text x="74" y="99" fill="#dbe4ee" font-size="12" text-anchor="middle">one review</text>
<g font-size="11.5">
<rect x="250" y="40" width="150" height="26" rx="6" fill="#1b2430" stroke="#2d3a4c"/><text x="262" y="58" fill="#dbe4ee">sentiment</text>
<rect x="250" y="74" width="150" height="26" rx="6" fill="#1b2430" stroke="#2d3a4c"/><text x="262" y="92" fill="#dbe4ee">emotion</text>
<rect x="250" y="108" width="150" height="26" rx="6" fill="#1b2430" stroke="#2d3a4c"/><text x="262" y="126" fill="#dbe4ee">irony</text>
<rect x="250" y="142" width="150" height="26" rx="6" fill="#1b2430" stroke="#2d3a4c"/><text x="262" y="160" fill="#dbe4ee">toxicity</text>
<rect x="440" y="40" width="150" height="26" rx="6" fill="#152130" stroke="#a6cf46"/><text x="452" y="58" fill="#a6cf46">Positive</text>
<rect x="440" y="74" width="150" height="26" rx="6" fill="#152130" stroke="#c98bdc"/><text x="452" y="92" fill="#e2c9f0">joy</text>
<rect x="440" y="108" width="150" height="26" rx="6" fill="#152130" stroke="#e0a24e"/><text x="452" y="126" fill="#e9c58a">ironic 95%</text>
<rect x="440" y="142" width="150" height="26" rx="6" fill="#152130" stroke="#a6cf46"/><text x="452" y="160" fill="#a6cf46">clean</text></g>
<g stroke="#3a4756" fill="none"><path d="M134 95 L250 53"/><path d="M134 95 L250 87"/><path d="M134 95 L250 121"/><path d="M134 95 L250 155"/></g>
<g fill="#e0a24e" font-size="13"><text x="410" y="57">&#8594;</text><text x="410" y="91">&#8594;</text><text x="410" y="125">&#8594;</text><text x="410" y="159">&#8594;</text></g>
<text x="345" y="184" fill="#61738a" font-size="11" text-anchor="middle" font-style="italic">they can disagree (Positive + ironic) &#8212; that disagreement is the lesson</text></svg>

> 🎓 **Class connection.** These are all *text classification* (Week 3) — the point is that the Hub has a specialist for almost any label scheme, ready to borrow.

> 🚀 **Final project.** Chaining several off-the-shelf classifiers is a legitimate, fast way to build a rich feature set or moderation layer — no training required.

In [ ]:
emotion = pipeline('text-classification', model='j-hartmann/emotion-english-distilroberta-base', top_k=None)
irony   = pipeline('text-classification', model='cardiffnlp/twitter-roberta-base-irony', top_k=None)
toxic   = pipeline('text-classification', model='unitary/toxic-bert', top_k=None)

def _flat(res):
    return res[0] if res and isinstance(res[0], list) else res
def _score(res, label):
    return next((d['score'] for d in _flat(res) if d['label'] == label), 0.0)

r = '10/10 would rage quit again.'
print('emotion:', _flat(emotion(r))[0]['label'],
      '| irony:', round(_score(irony(r), 'irony'), 2),
      '| toxic:', round(_score(toxic(r), 'toxic'), 2))

## 7 · Analyze a whole game — voice of customer

**What this cell does.** Runs the Router across a sample of *all* of a game's reviews and averages the scores — turning one-review-at-a-time classification into an aggregate **topic breakdown**, then plots it.

**Why aggregate.** One review is an anecdote; the distribution over hundreds is a finding. This is the jump from *classifying* to *measuring*.

> 🎓 **Class connection.** This is the *document-collection → theme distribution* idea that Thursday's topic-modeling lecture makes its whole focus — here done with a classifier instead of an unsupervised model.

> 🚀 **Final project.** Aggregating model outputs into a chart is what turns a demo into an **insight** — a voice-of-customer dashboard is a compelling final deliverable.

In [ ]:
TOPIC_LABELS = ['difficulty', 'story', 'graphics', 'price or value', 'bugs or performance', 'multiplayer']
_AGG = {}
def analyze_game(game, n=24):
    if game in _AGG: return _AGG[game]
    revs = df[df['game'] == game]['review'].sample(min(n, (df['game']==game).sum()),
                                                    random_state=0).str.slice(0, 400).tolist()
    res = router(revs, candidate_labels=TOPIC_LABELS, multi_label=True)
    res = res if isinstance(res, list) else [res]
    totals = {l: 0.0 for l in TOPIC_LABELS}
    for one in res:
        for l, s in zip(one['labels'], one['scores']): totals[l] += s
    out = sorted(((l, v/len(res)) for l, v in totals.items()), key=lambda x: -x[1])
    _AGG[game] = out
    return out

import matplotlib.pyplot as plt
g = 'Cyberpunk 2077'
labs, vals = zip(*analyze_game(g))
plt.figure(figsize=(7,3)); plt.barh(labs[::-1], vals[::-1], color='#57b0e6')
plt.title(f'What {g} reviews are about'); plt.xlabel('avg zero-shot score'); plt.tight_layout(); plt.show()

## 8 · Wire it together — the assistant

**What this cell does.** Defines `assistant()` — the brain — plus the helpers that render each answer as a Steam-styled HTML card. It **reads the message and routes** to the right tool, then returns rich HTML (the same markup the chat app shows).

**The pattern (a router/controller).** A useful assistant isn't one model — it's a dispatcher that picks the right tool per input:

<svg viewBox="0 0 690 200" width="100%" style="max-width:690px;font-family:system-ui">
<rect width="690" height="200" rx="12" fill="#0f151d"/>
<text x="16" y="24" fill="#8ea0b3" font-size="12" font-family="monospace" letter-spacing="1.5">THE ASSISTANT = A ROUTER (your app's controller)</text>
<rect x="14" y="82" width="128" height="46" rx="8" fill="#1b2430" stroke="#57b0e6"/><text x="78" y="109" fill="#dbe4ee" font-size="12" text-anchor="middle">your message</text>
<g font-size="12" fill="#dbe4ee">
<rect x="330" y="40" width="270" height="34" rx="7" fill="#152130" stroke="#a6cf46"/><text x="344" y="62">retrieve &#8594; QA &#8594; recommend gauge</text>
<rect x="330" y="86" width="270" height="34" rx="7" fill="#152130" stroke="#57b0e6"/><text x="344" y="108">multi-lens + zero-shot Router tags</text>
<rect x="330" y="132" width="270" height="34" rx="7" fill="#152130" stroke="#c98bdc"/><text x="344" y="154">aggregate the whole game (VoC)</text></g>
<g stroke="#3a4756" fill="none"><path d="M142 100 L330 57"/><path d="M142 105 L330 103"/><path d="M142 110 L330 149"/></g>
<g font-size="10.5" fill="#8ea0b3"><text x="196" y="66">a question?</text><text x="196" y="99">a pasted review?</text><text x="196" y="150">"analyze &lt;game&gt;"?</text></g>
<text x="345" y="192" fill="#61738a" font-size="11" text-anchor="middle" font-style="italic">read the input, pick the right tool &#8212; the same shape your final-project demo needs</text></svg>

> 🎓 **Class connection.** Everything from tonight converges here: retrieval + QA + zero-shot + the lens models, composed behind one function.

> 🚀 **Final project.** This routing function *is* your app's controller. Whatever your project does, it needs this layer — parse the request, call the right model(s), format the response.

In [ ]:
from IPython.display import HTML, display
import math

QUESTION_STARTS = {'what','why','how','is','are','does','do','can','should','which','when','who','worth'}
DEFAULT_LABELS  = 'difficulty, story, graphics, price or value, bugs or performance, multiplayer'
BAR_COLORS = ['#57b0e6', '#a6cf46', '#c98bdc', '#e0a24e', '#5f7183', '#5f7183']
EMO_EMOJI  = {'anger':'&#128544;','joy':'&#128516;','sadness':'&#128546;','fear':'&#128552;','surprise':'&#128562;','disgust':'&#129314;','neutral':'&#128528;'}
_KEYS = '<style>@keyframes sypgrow{from{width:0 !important}}</style>'
WALL, LAST = [], {}          # WALL is shared across all users of the running app

def looks_like_question(t):
    t = t.strip().lower()
    return t.endswith('?') or (t.split() and t.split()[0] in QUESTION_STARTS)

def _card(inner):
    return (_KEYS + '<div style="background:#151d27;border:1px solid #243040;border-radius:13px;'
            'padding:13px 15px;font-family:system-ui,sans-serif;max-width:560px;">' + inner + '</div>')

def _fill(pct, color):
    return ('<span style="height:7px;background:#0e1620;border:1px solid #243040;border-radius:99px;'
            'display:block;overflow:hidden;">'
            f'<span style="display:block;height:100%;width:{pct}%;background:{color};'
            'animation:sypgrow .7s ease-out;"></span></span>')

def _gauge(pct, color):
    r = 22; c = 2*math.pi*r; off = c*(1-pct/100)
    return (f'<svg width="58" height="58" viewBox="0 0 58 58" style="flex:none;">'
            f'<circle cx="29" cy="29" r="{r}" fill="none" stroke="#22303f" stroke-width="6"/>'
            f'<circle cx="29" cy="29" r="{r}" fill="none" stroke="{color}" stroke-width="6" stroke-linecap="round" '
            f'stroke-dasharray="{c:.1f}" stroke-dashoffset="{off:.1f}" transform="rotate(-90 29 29)"/>'
            f'<text x="29" y="33" text-anchor="middle" font-family="monospace" font-size="13" '
            f'font-weight="700" fill="{color}">{pct}%</text></svg>')

def _tag_rows(pairs):
    rows = ''
    for i, (lab, sc) in enumerate(pairs):
        rows += ('<div style="display:grid;grid-template-columns:150px 1fr 42px;align-items:center;gap:10px;margin:7px 0;">'
                 f'<span style="font-size:13px;color:#dbe4ee;">{lab}</span>'
                 + _fill(int(round(sc*100)), BAR_COLORS[i % len(BAR_COLORS)]) +
                 f'<span style="font-family:monospace;font-size:11.5px;color:#93a4b7;text-align:right;">{sc:.2f}</span></div>')
    return rows

def _label(text, color):
    return (f'<div style="font-family:monospace;font-size:10.5px;letter-spacing:1.2px;'
            f'text-transform:uppercase;color:{color};margin:2px 0 8px;">{text}</div>')

def _chip(name, value, color):
    return ('<span style="display:inline-flex;gap:6px;align-items:center;background:#0f1720;'
            'border:1px solid #22303f;border-radius:8px;padding:5px 9px;margin:0 6px 6px 0;font-size:12px;">'
            f'<span style="color:#7a8b9a;">{name}</span>'
            f'<span style="color:{color};font-weight:600;">{value}</span></span>')

def _lens_html(text):
    t = text[:512]
    s = sentiment(t)[0]; scol = '#a6cf46' if s['label'] == 'POSITIVE' else '#e5705f'
    emo = _flat(emotion(t))[0]
    iro = _score(irony(t), 'irony'); tox = _score(toxic(t), 'toxic')
    chips = (_chip('sentiment', s['label'].title(), scol)
             + _chip('emotion', EMO_EMOJI.get(emo['label'], '') + ' ' + emo['label'], '#c98bdc')
             + _chip('irony', ('&#128681; %d%%' % round(iro*100)) if iro > 0.5 else ('%d%%' % round(iro*100)),
                     '#e0a24e' if iro > 0.5 else '#7a8b9a')
             + _chip('toxicity', ('&#128681; %d%%' % round(tox*100)) if tox > 0.5 else 'clean &#10003;',
                     '#e5705f' if tox > 0.5 else '#a6cf46'))
    return _label('Multi-lens &middot; four models, one review', '#57b0e6') + '<div>' + chips + '</div>'

def _qa_html(game, answer, score):
    verdict = ''
    if game is not None:
        g = df[df['game'] == game]
        pct = int(round((g['recommended'] == 'Recommended').mean() * 100))
        gcol = '#a6cf46' if pct >= 60 else '#e0a24e' if pct >= 45 else '#e5705f'
        art = (f'<img src="{cover(game)}" alt="" style="width:120px;border-radius:8px;'
               'border:1px solid #2d3a4c;flex:none;" onerror="this.style.display=&quot;none&quot;">'
               if cover(game) else '')
        verdict = ('<div style="display:flex;align-items:center;gap:12px;padding:10px;border-radius:10px;'
                   'background:#0f1720;border:1px solid #22303f;margin-bottom:11px;">'
                   + art + _gauge(pct, gcol) +
                   f'<span style="font-size:12.5px;color:#c8d3df;"><b style="color:#eaf2fb;">{game}</b><br>'
                   f'{pct}% of {len(g)} reviewers recommend it</span></div>')
    ans = ('<div style="font-size:13.5px;color:#dbe4ee;line-height:1.5;">From the reviews: '
           f'<mark style="background:#123049;color:#bfe2fb;padding:1px 6px;border-radius:5px;">{answer}</mark></div>')
    meter = ('<div style="display:flex;align-items:center;gap:9px;margin-top:11px;">'
             '<span style="font-size:11.5px;color:#93a4b7;">answer confidence</span>'
             + _fill(int(round(score*100)), '#57b0e6') +
             f'<span style="font-family:monospace;font-size:11.5px;color:#93a4b7;">{score:.2f}</span></div>')
    return _card(_label('Ask the reviews', '#57b0e6') + verdict + ans + meter)

def assistant(message, history=None, labels=DEFAULT_LABELS):
    msg = message.strip()
    if msg.lower().startswith('analyze '):
        game = detect_game(msg[8:]) or detect_game(msg)
        if game:
            return _card(_label(f'What {game} reviews are about', '#a6cf46') + _tag_rows(analyze_game(game)))
        return _card('<span style="color:#e5705f">Name a game I have, e.g. <b>analyze Elden Ring</b>.</span>')
    if looks_like_question(msg):
        game = detect_game(msg)
        ctx = ' '.join(top_reviews(msg, k=3, game=game))[:2000]
        a = qa(question=msg, context=ctx)
        return _qa_html(game, a['answer'], a['score'])
    labs = [l.strip() for l in (labels or DEFAULT_LABELS).split(',') if l.strip()]
    r = router(msg[:400], candidate_labels=labs, multi_label=True)
    tags = _label('Review Router &middot; zero-shot tags', '#a6cf46') + _tag_rows(list(zip(r['labels'], r['scores']))[:5])
    return _card(_lens_html(msg) + '<div style="height:8px"></div>' + tags)

display(HTML(assistant('Is Elden Ring worth it?')))
display(HTML(assistant('Great game if you enjoy losing your entire weekend.')))

## 9 · The chat app — the whole class joins

**What this cell does.** Wraps the assistant in a **Gradio** web app — a dark Steam-styled chat with cover art, a recommend gauge, quick-pick buttons, an editable labels box, and a shared *Confidently-Wrong Wall*. `launch(share=True)` prints a public URL anyone can open. *(The wiring auto-adapts to Gradio 5 or 6.)*

**Try to break it.** Ask a game question, paste a review, type **analyze &lt;game&gt;**, hit a 😈 trap, and **flag** any confident-wrong answer to the Wall.

> 🎓 **Class connection.** A model with no interface is a lab result; wrapping it in a UI is what makes it a *product* — the throughline of the whole course.

> 🚀 **Final project.** Gradio (or Streamlit, Week 8) is exactly how you'll ship your **final demo**. This cell is a working template — swap in your `assistant()` and you have a shareable app.

In [ ]:
import gradio as gr

QUICKPICKS = ['Elden Ring', 'Sekiro', 'Cyberpunk 2077', 'Hades', "Baldur's Gate 3", 'Stardew Valley']
TRAPS = ['10/10 would rage quit again.', "It's not a bad game.",
         'Great game if you enjoy losing your entire weekend.']

CSS = """
.gradio-container{background:#0c1118 !important;}
footer{display:none !important;}
#syp-header{display:flex;align-items:center;gap:14px;padding:12px 4px 8px;}
#syp-header .mark{width:38px;height:38px;border-radius:10px;background:#0e1a26;border:1px solid #2d3a4c;
  display:grid;place-items:center;color:#57b0e6;font-size:20px;}
#syp-header .ttl{font-size:17px;font-weight:650;color:#dbe4ee;}
#syp-header .sub{font-size:12px;color:#93a4b7;}
#syp-header .live{margin-left:auto;font-family:ui-monospace,monospace;font-size:11.5px;color:#93a4b7;
  background:#0e1620;border:1px solid #243040;padding:6px 11px;border-radius:999px;}
"""

theme = gr.themes.Base(primary_hue='blue', neutral_hue='slate').set(
    body_background_fill='#0c1118', block_background_fill='#151d27',
    block_border_color='#243040', body_text_color='#dbe4ee',
    input_background_fill='#0d141d', input_border_color='#2d3a4c',
    button_primary_background_fill='#57b0e6', button_primary_text_color='#052033',
)

HEADER = ('<div id="syp-header"><div class="mark">&#127918;</div>'
          '<div><div class="ttl">Should You Play It?</div>'
          '<div class="sub">Steam review assistant &middot; ISBA 2411</div></div>'
          f'<div class="live">&#9679; live &middot; {len(df):,} reviews &middot; {df.game.nunique()} games</div></div>')

def render_wall():
    if not WALL:
        return '<div style="color:#5f7183;font-size:13px;">No flagged answers yet — go break the bot.</div>'
    items = ''.join(
        f'<div style="border:1px solid #3a1712;background:#1c1210;border-radius:9px;padding:9px 11px;margin:6px 0;">'
        f'<div style="color:#e5705f;font-size:12px;font-family:monospace;">#{i+1}</div>'
        f'<div style="color:#dbe4ee;font-size:13px;">{w["user"]}</div></div>'
        for i, w in enumerate(WALL[-12:]))
    return f'<div style="font-family:monospace;font-size:11px;color:#e5705f;margin-bottom:6px;">CONFIDENTLY-WRONG WALL &middot; {len(WALL)}</div>' + items

def respond(message, chat_history, labels_val):
    if not message.strip():
        yield '', chat_history; return
    ch = (chat_history or []) + [{'role': 'user', 'content': message}]
    yield '', ch + [{'role': 'assistant', 'content': '<span style="color:#93a4b7">…reading the reviews</span>'}]
    reply = assistant(message, labels=labels_val)
    LAST['user'] = message; LAST['reply'] = reply
    yield '', ch + [{'role': 'assistant', 'content': reply}]

def flag():
    if LAST.get('user'): WALL.append(dict(LAST))
    return render_wall()

# version-adaptive wiring: Gradio 6 dropped Chatbot(type=) and moved theme/css to launch()
_GMAJ = int(gr.__version__.split('.')[0])
_chat_kw = dict(height=430, sanitize_html=False, show_label=False)
if _GMAJ < 6:
    _chat_kw['type'] = 'messages'
_blocks_kw = dict(title='Should You Play It?')
_launch_kw = dict(share=True)
(_blocks_kw if _GMAJ < 6 else _launch_kw).update(theme=theme, css=CSS)

with gr.Blocks(**_blocks_kw) as demo:
    gr.HTML(HEADER)
    chat = gr.Chatbot(**_chat_kw)
    with gr.Row():
        picks = [gr.Button(g, size='sm') for g in QUICKPICKS]
    with gr.Row():
        traps = [gr.Button('😈 ' + t[:22] + '…', size='sm') for t in TRAPS]
    with gr.Row():
        msg  = gr.Textbox(placeholder='Ask about a game, paste a review, or "analyze <game>"...',
                          show_label=False, scale=5)
        send = gr.Button('Send', variant='primary', scale=1)
    labels = gr.Textbox(value=DEFAULT_LABELS, label='Router labels (invent your own!)')
    with gr.Row():
        flag_btn = gr.Button('🚩 Flag last answer as confidently wrong', size='sm')
        refresh  = gr.Button('Refresh wall', size='sm')
    wall = gr.HTML(render_wall())

    send.click(respond, [msg, chat, labels], [msg, chat])
    msg.submit(respond, [msg, chat, labels], [msg, chat])
    for b, g in zip(picks, QUICKPICKS):
        b.click(lambda g=g: f'Is {g} worth it?', None, msg)
    for b, t in zip(traps, TRAPS):
        b.click(lambda t=t: t, None, msg)
    flag_btn.click(flag, None, wall)
    refresh.click(render_wall, None, wall)

demo.launch(**_launch_kw)

## 10 · Put a QR on the screen
Paste the **public URL** the cell above printed (the `https://….gradio.live` one) and run this to show a QR the room can scan.

In [ ]:
import qrcode
SHARE_URL = 'https://your-link.gradio.live'   # <-- paste the printed public URL
qrcode.make(SHARE_URL)

### Fallback (if campus wi-fi blocks the public link)
Same brain, inline — no public URL.

In [ ]:
import ipywidgets as w
from IPython.display import display, HTML
box = w.Text(placeholder='Ask about a game, or paste a review...', layout=w.Layout(width='70%'))
go = w.Button(description='Send', button_style='primary'); out = w.Output()
def handle(_):
    if not box.value.strip(): return
    ans = assistant(box.value)
    with out: display(HTML(f'<div style="color:#93a4b7;font-size:12px;margin-top:8px;">you: '
                           f'{box.value}</div>' + ans))
    box.value = ''
go.on_click(handle)
display(w.HBox([box, go]), out)

---
## Take-home exercise
1. **Router:** pick 15 reviews and a label set of your own. Find two the Router gets wrong — name why.
2. **Multi-lens:** find a review where the four lenses *disagree* (e.g. sentiment POSITIVE but irony flagged). Which lens do you trust, and why?
3. **QA:** ask five game questions. Find one it nails and one it fumbles — name the failure mode.
4. **Aggregate:** `analyze` two games. Does the topic mix match the game's reputation?

Hand in a short write-up: approach, findings, three failure cases. *(Maps to the midterm rubric's Analysis & Insight.)*